In [1]:
# %%
# -*- coding: utf-8 -*-
"""
Constitutional Question Organizer + LLM Processing (Stateful)

This script processes questions in "levels" based on a dependency plan
generated by `dependency_scheduler_levels.py`.

It maintains a "state" of answers from previous levels and feeds
them to the LLM as context for answering dependent questions.
"""

import re
import time
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI
from typing import Tuple, Dict, Any, List, Set

class ConstitutionalQuestionOrganizer:
    def __init__(self, questions_json_path: str = "organized_constitutional_questions.json") -> None:
        # Load both chunks (for metadata) and the flat question map (for lookup)
        self.question_chunks, self.all_questions_map, self.code_to_chunk_map = self._load_organized_questions(questions_json_path)

    def _load_organized_questions(self, json_path: str) -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, str]]:
        """Load organized questions from JSON output of first script."""
        json_file = Path(json_path)
        if not json_file.exists():
            raise FileNotFoundError(f"Organized questions file not found: {json_path}")
        
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        chunks = data.get("chunks", {})
        
        all_questions_map = {}
        code_to_chunk_map = {}
        for chunk_id, chunk_data in chunks.items():
            for q in chunk_data.get("questions", []):
                identifier = q.get("code") or q["id"]
                if identifier:
                    all_questions_map[identifier] = q
                    code_to_chunk_map[identifier] = chunk_id
                    
        return chunks, all_questions_map, code_to_chunk_map

    def get_question_by_code(self, code: str) -> Dict[str, Any]:
        """Fetches full question data by its code or ID."""
        return self.all_questions_map.get(code, {})

    def get_chunk_data(self, chunk_id: str) -> Dict[str, Any]:
        """Fetches metadata (title, desc) for a chunk."""
        return self.question_chunks.get(chunk_id, {})
    
    def create_chunk_prompt(
                self, 
                constitutional_text: str, 
                chunk_title: str,
                chunk_description: str,
                questions_for_this_prompt: List[Dict[str, Any]],
                answer_context: Dict[str, str], 
                country: str, 
                year: str
            ) -> str:
                """Create a structured prompt with Few-Shot Examples and Chain-of-Thought instructions."""
                
                question_blocks = []
                for i, q in enumerate(questions_for_this_prompt, 1):
                    # Build the question block (same as before)
                    block = f"{i}. [{q.get('code') or q['id']}] {q.get('question', '')}"
                    
                    conditional_dict = q.get("conditional", {})
                    raw_cond_text = conditional_dict.get("raw")
                    if raw_cond_text:
                        block += f"\n   CONDITION: {raw_cond_text}"

                    if q.get("multi_select"):
                        block += "\n   (Multiple selections allowed)"

                    if q.get("options"):
                        for opt in q["options"]:
                            block += f"\n   {opt['number']}. {opt['text']}"

                    if q.get("instructions"):
                        block += f"\n   Instructions: {q['instructions']}"

                    question_blocks.append(block)

                question_text = "\n\n".join(question_blocks)

                # --- Context Block (Same as before) ---
                context_block = ""
                if answer_context:
                    context_lines = ["### Context: Previously Answered Questions ###",
                                    "Use these answers to resolve dependencies for the new questions.",
                                    "Do NOT re-answer these questions."]
                    for code, answer in answer_context.items():
                        q_data = self.get_question_by_code(code)
                        q_text = q_data.get('question', 'Unknown Question')
                        answer_text = f"Option {answer}" 
                        if q_data.get('options'):
                            for opt in q_data['options']:
                                if str(opt['number']) == str(answer):
                                    answer_text = f"{answer} ({opt['text']})"
                                    break
                        context_lines.append(f"- {code} ({q_text}): {answer_text}")
                    context_block = "\n".join(context_lines) + "\n"

                # --- Construct Final Prompt ---
                prompt = f"""
                You are analyzing the constitution of {country} ({year}).

                {context_block} 

                ### Thematic Area: {chunk_title}
                {chunk_description}

                ### Questions to Answer ({len(questions_for_this_prompt)} total):
                {question_text}

                ### Constitutional Text:
                {constitutional_text}

                ### INSTRUCTIONS ###
                1. **Evidence Only:** Answer ONLY based on the text provided. Do not infer. If the topic is not mentioned, use the 'Not Specified' code (98).
                2. **Formatting (CRITICAL):** For EVERY question, you must use this exact two-line format:
                ANALYSIS: [One sentence citing the specific Article/Section and justifying the choice]
                FINAL: [CODE]|[OPTION_NUMBER]

                ### EXAMPLES ###

                Example 1: Explicit Right
                Text: "All citizens shall have the right to free speech."
                Question: [v12] Freedom of speech?
                1. Yes  2. No  98. Not Specified
                ANALYSIS: The text in Article 3 explicitly grants the right to free speech.
                FINAL: v12|1

                Example 2: Multi-Select (Select All That Apply)
                Text: "Amendments may be proposed by the President or by either chamber of the Legislature."
                Question: [v74] Who can propose amendments?
                1. Head of State  2. Head of Government  4. First Chamber  5. Second Chamber
                ANALYSIS: Article 100 identifies the President (Head of State) and both legislative chambers as having the power to propose amendments. [cite: 1]
                FINAL: v74|1, 4, 5

                Example 3: Explicit Denial
                Text: "The President may not dissolve Parliament."
                Question: [v55] Can the Executive dissolve the Legislature?
                1. Yes  2. No  98. Not Specified
                ANALYSIS: The text explicitly forbids the President from dissolving Parliament.
                FINAL: v55|2

                Example 4: Silence (Not Specified)
                Text: "The President is elected for 5 years."
                Question: [v99] Is there a Vice President?
                1. Yes  2. No  98. Not Specified
                ANALYSIS: The text mentions the President but is completely silent on the office of a Vice President. I will not infer one exists.
                FINAL: v99|98

                **Your Task:**
                Provide the analysis and final codes for the {len(questions_for_this_prompt)} questions listed above.
                """
                return prompt.strip()

# (The rest of the file, including parse_llm_answer, process_constitutions,
# convert_json_dir_to_csv, and the __main__ block, remains unchanged
# as its logic is compatible with this update.)
import re
from typing import Dict

def parse_llm_answer(answer_text: str) -> Dict[str, str]:
    """
    Parses the LLM response using the "Reasoning First" format.
    
    Expected Format from LLM:
      ANALYSIS: The text explicitly states...
      FINAL: [CODE]|[OPTION]
    """
    answers = {}
    if not answer_text:
        return answers

    # Regex to capture the pattern: "FINAL: <CODE>|<OPTION>"
    # It handles optional whitespace and case insensitivity for "FINAL:"
    # Group 1 = Question Code (e.g., v12 or AMEND)
    # Group 2 = Option Number(s) (e.g., 1 or 1,2)
    pattern = r"FINAL:\s*([A-Za-z0-9_]+)\|([0-9,\s]+)"
    
    matches = re.findall(pattern, answer_text, re.IGNORECASE)
    
    for code, val in matches:
        # Clean up the extracted values
        clean_code = code.strip()
        clean_val = val.strip()
        answers[clean_code] = clean_val
        
    return answers


def process_constitutions(
    csv_path: str, 
    api_key: str,
    questions_json_path: str = "organized_constitutional_questions_v4.json",
    dependency_plan_path: str = "dependency_plan_levels.json", # <-- USE NEW PLAN
    output_dir: str = "llm_outputs",
    temp: int = 0,
    seed: int = 123
) -> None:
    """Run LLM analysis across all constitutions, level-by-level."""

    df = pd.read_csv(csv_path)
    organizer = ConstitutionalQuestionOrganizer(questions_json_path=questions_json_path)
    
    plan_path = Path(dependency_plan_path)
    if not plan_path.exists():
        raise FileNotFoundError(f"Dependency plan file not found: {dependency_plan_path}")
    with open(plan_path, "r", encoding="utf-8") as f:
        dependency_plan: List[List[str]] = json.load(f) # List of levels
    
    client = OpenAI(api_key=api_key)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    start_total = time.time()

    for _, row in df.iterrows():
        constitution_text = row["content"]
        name = row["name"]
        name_no_ext = Path(name).stem
        parts = name_no_ext.split("_")
        country, year = (" ".join(parts[:-1]), parts[-1]) if parts and parts[-1].isdigit() else (name_no_ext, "Unknown")

        # --- STATEFUL PROCESSING ---
        # This holds all answers (CODE: "option") as we go.
        global_answer_context: Dict[str, str] = {} 
        # This will store the final JSON output, grouped by original chunk
        final_results_by_chunk: Dict[str, Any] = {}
        # --- END STATEFUL ---

        start_constitution = time.time()
        total_tokens = 0
        
        print(f"\nProcessing {country} ({year})...")

        for level_index, level_codes in enumerate(dependency_plan):
            print(f"  - Processing Level {level_index} ({len(level_codes)} questions)...")
            
            # Group questions in this level by their original chunk
            # This keeps prompts thematically coherent
            questions_grouped_by_chunk: Dict[str, List[Dict[str, Any]]] = {}
            for code in level_codes:
                chunk_id = organizer.code_to_chunk_map.get(code)
                if not chunk_id:
                    print(f"Warning: Code '{code}' not in code_to_chunk_map. Skipping.")
                    continue
                
                question_data = organizer.get_question_by_code(code)
                if not question_data:
                    print(f"Warning: Code '{code}' not in all_questions_map. Skipping.")
                    continue
                
                if chunk_id not in questions_grouped_by_chunk:
                    questions_grouped_by_chunk[chunk_id] = []
                questions_grouped_by_chunk[chunk_id].append(question_data)

            # Now, process one "sub-prompt" per chunk *within* this level
            for chunk_id, questions_in_chunk in questions_grouped_by_chunk.items():
                
                chunk_metadata = organizer.get_chunk_data(chunk_id)
                
                prompt = organizer.create_chunk_prompt(
                    constitutional_text=constitution_text,
                    chunk_title=chunk_metadata.get("title", "Untitled"),
                    chunk_description=chunk_metadata.get("description", ""),
                    questions_for_this_prompt=questions_in_chunk,
                    answer_context=global_answer_context, # Pass in all answers so far
                    country=country,
                    year=year,
                )

                t0 = time.time()
                completion = client.chat.completions.create(
                    # model="gpt-4o-mini",
                    # model="gpt-4o",
                    model="gpt-5.1",
                    messages=[
                    # Inside process_constitutions, update the messages list:
                        {
                            "role": "system", 
                            "content": (
                                "You are an expert data entry assistant for constitutional analysis. "
                                "Your goal is to extract specific legal data with 100% accuracy. "
                                "1. Answer based EXCLUSIVELY on the provided ### Constitutional Text ###. "
                                "2. If the answer is not in the text, you must select the numeric option for 'Not Specified' (usually 98). "
                                "3. Ignore all external knowledge. "
                                "4. Follow the formatting instructions in the user prompt exactly to ensure the data can be parsed."
                            )
                        },
                        {"role": "user", "content": prompt}
                    ],
                    temperature=temp,
                    seed=seed
                )
                t1 = time.time()

                answer_str = completion.choices[0].message.content
                parsed_answers = parse_llm_answer(answer_str)
                
                # --- UPDATE STATE AND FINAL RESULTS ---
                tokens = completion.usage.total_tokens if hasattr(completion, "usage") else 0
                total_tokens += tokens

                # Ensure the chunk exists in the final output
                if chunk_id not in final_results_by_chunk:
                    final_results_by_chunk[chunk_id] = {
                        "title": chunk_metadata.get("title"),
                        "questions_shown": [],
                        "answers_received": [],
                        "time_sec": 0,
                        "tokens_used": 0
                    }
                
                # Store results for final JSON
                final_results_by_chunk[chunk_id]["questions_shown"].extend(questions_in_chunk)
                final_results_by_chunk[chunk_id]["answers_received"].append(answer_str)
                final_results_by_chunk[chunk_id]["time_sec"] += round(t1 - t0, 2)
                final_results_by_chunk[chunk_id]["tokens_used"] += tokens
                
                # CRITICAL: Update the global context for the next level
                for code, answer_val in parsed_answers.items():
                    # We only need the option number for logic, not open-ended text
                    option_number = answer_val.split('|')[0].strip()
                    global_answer_context[code] = option_number 
                # --- END UPDATE ---

        # --- END OF ALL LEVELS ---
        
        # Consolidate answers for the final JSON
        final_results = {}
        for chunk_id, data in final_results_by_chunk.items():
            final_results[chunk_id] = {
                "title": data["title"],
                "questions_shown": data["questions_shown"],
                "answer": "\n".join(data["answers_received"]), # Combine all answers for this chunk
                "time_sec": data["time_sec"],
                "tokens_used": data["tokens_used"]
            }

        total_time = round(time.time() - start_constitution, 2)

        # Save one JSON per constitution
        out_file = out_dir / f"{country}_{year}.json"
        output_data = {
            "country": country,
            "year": year,
            "results": final_results, # This is the chunk-grouped data
            "metadata": {
                "total_time_sec": total_time,
                "total_tokens": total_tokens,
                "processing_levels": len(dependency_plan)
            },
            # "final_answer_context": global_answer_context # Optional: for debugging
        }
        
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)

        print(f"Processed {country}_{year} in {total_time}s, tokens={total_tokens}")

    end_total = time.time()
    print(f"\nTotal runtime for all constitutions: {round(end_total - start_total, 2)} seconds")


# JSON-to-CSV converter
# -*- coding: utf-8 -*-
import sys
import pandas as pd
import json
import re
from pathlib import Path

def convert_json_dir_to_csv(
    json_dir: str = "llm_outputs_explanations",
    output_csv_original: str = "Port_Taiwan_US_original.csv",
    output_csv_dummy: str = "Port_Taiwan_US_dummy.csv",
    output_csv_pivot: str = "Port_Taiwan_US_pivot.csv"
) -> None:
    """
    Scans a directory for JSON files and creates three separate CSVs.
    
    Updated to handle Chain-of-Thought responses formatted as:
      ANALYSIS: [Reasoning text...]
      FINAL: [CODE]|[ANSWER]
    """
    input_path = Path(json_dir)
    if not input_path.is_dir():
        print(f"Error: Directory not found at '{json_dir}'")
        return

    all_rows_original = [] 
    all_rows_dummy = []    

    json_files = list(input_path.glob("*.json"))
    if not json_files:
        print(f"No JSON files found in '{json_dir}'.")
        return

    print(f"Found {len(json_files)} JSON files to process...")

    # Regex to find the answer line. 
    # Group 1 = Code, Group 2 = Answer Values
    answer_pattern = re.compile(r"FINAL:\s*([A-Za-z0-9_]+)\|([0-9,\s]+)", re.IGNORECASE)

    for json_file in json_files:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        country = data.get("country", "Unknown")
        year = data.get("year", "Unknown")
        results = data.get("results", {})

        # -------------------------------------------------------
        # 1. Build Question Lookup Map (Metadata)
        # -------------------------------------------------------
        for chunk_id, chunk_data in results.items():
            question_lookup = {}
            
            # Pre-process questions to have all metadata ready
            for q_data in chunk_data.get("questions_shown", []):
                q_id = q_data.get("id")
                q_code = q_data.get("code")
                
                if not (q_id or q_code): continue
                
                q_options_list = q_data.get("options", [])
                
                # Determine if this question should be dummified (exploded into 0/1 vars)
                has_option_codes = any(opt.get("code") for opt in q_options_list)
                is_multi_select = q_data.get("multi_select", False)
                is_dummifiable = has_option_codes or is_multi_select
                
                # Map option numbers to text for standard answers
                options_map_for_lookup = {str(opt.get("number")): opt.get("text", "") for opt in q_options_list}

                # Store by Code OR ID to ensure we can find it
                key = q_code or q_id
                question_lookup[key] = {
                    "id": q_id,
                    "code": q_code,
                    "question_text": q_data.get("question", ""),
                    "options_list": q_options_list,
                    "options_map": options_map_for_lookup,
                    "conditional": q_data.get("conditional"),
                    "instructions": q_data.get("instructions"),
                    "order_index": q_data.get("order_index", 99999),
                    "is_dummifiable": is_dummifiable
                }

            # -------------------------------------------------------
            # 2. Parse The LLM Response Text
            # -------------------------------------------------------
            answer_string = chunk_data.get("answer", "")
            if not answer_string.strip(): continue

            time_sec = chunk_data.get("time_sec", None)
            tokens_used = chunk_data.get("tokens_used", None)

            # We split the text by the Answer Pattern to isolate the Reasoning text before it
            # re.split with capturing groups returns [text_before, group1, group2, text_between, group1, ...]
            parts = re.split(answer_pattern, answer_string)
            
            # Iterate through the split parts
            # parts[0] is text before first match (reasoning for Q1)
            # parts[1] is Q1 Code
            # parts[2] is Q1 Answer
            # parts[3] is text before second match (reasoning for Q2)...
            
            current_explanation = ""
            
            # Loop start at index 0. Steps of 3 because (text, code, val)
            i = 0
            while i < len(parts):
                # The text chunk is the reasoning for the *next* question found
                text_chunk = parts[i]
                
                # Clean up the explanation text
                clean_explanation = text_chunk.replace("ANALYSIS:", "").strip()
                
                # If there are enough parts remaining, the next two are Code and Answer
                if i + 2 < len(parts):
                    q_code = parts[i+1].strip()
                    answer_code_raw = parts[i+2].strip()
                    
                    # Now process this question answer pair
                    q_info = question_lookup.get(q_code, {})
                    if not q_info: 
                        i += 3
                        continue 

                    # Extract Metadata
                    q_id = q_info.get("id", "NA")
                    q_code_clean = q_info.get("code", q_code)
                    order_index_val = q_info.get("order_index", 99999)
                    question_text = q_info.get("question_text", "NA")
                    instructions_text = q_info.get("instructions", "NA")
                    
                    # Handle Conditional Logic string extraction
                    conditional_data = q_info.get("conditional", "NA")
                    conditional_var = "NA"
                    if isinstance(conditional_data, dict) and conditional_data.get("raw"):
                        conditional_var = conditional_data.get("raw").strip(" ()")

                    # -------------------------------------------------------
                    # 3. Create Rows (Standard vs Dummy)
                    # -------------------------------------------------------
                    dummify_this_question = q_info.get("is_dummifiable", False)

                    if dummify_this_question:
                        # --- DUMMY LOGIC (File 2 only) ---
                        selected = {a.strip() for a in answer_code_raw.split(",")}
                        
                        for opt in q_info.get("options_list", []):
                            opt_num = str(opt.get("number"))
                            opt_text = opt.get("text")
                            opt_code = opt.get("code")
                            # Determine Variable Name
                            variable_name = opt_code or f"{q_code_clean}_{opt_num}"
                            
                            is_selected = (opt_num in selected)

                            all_rows_dummy.append({
                                "country": country, "year": year, "chunk_id": chunk_id,
                                "variable_name": variable_name, 
                                "question_id": q_id, "question_code": q_code_clean,
                                "question_text": question_text,
                                "conditional_variable": conditional_var,
                                "instructions_text": instructions_text,
                                "answer_code": opt_num, "answer_text": opt_text,
                                "explanation": clean_explanation if is_selected else "", # Only explain selected
                                "chunk_time_sec": time_sec,
                                "total_chunk_tokens": tokens_used,
                                "value": 1 if is_selected else 0,
                                "order_index": order_index_val
                            })
                    else:
                        # --- STANDARD LOGIC (File 1 & 2) ---
                        answer_text_final = q_info.get("options_map", {}).get(answer_code_raw, "")
                        
                        # Safety cast to int
                        try: numeric_value = int(answer_code_raw)
                        except ValueError: numeric_value = 0 

                        # Row for Original File
                        all_rows_original.append({
                            "country": country, "year": year, "chunk_id": chunk_id,
                            "variable_name": q_code_clean or q_id,
                            "question_id": q_id, "question_code": q_code_clean,
                            "question_text": question_text,
                            "conditional_variable": conditional_var,
                            "instructions_text": instructions_text,
                            "value": answer_code_raw, # String representation
                            "answer_text": answer_text_final,
                            "explanation": clean_explanation,
                            "chunk_time_sec": time_sec,
                            "total_chunk_tokens": tokens_used,
                            "order_index": order_index_val
                        })
                        
                        # Row for Master Dummy File (Unpivoted)
                        all_rows_dummy.append({
                            "country": country, "year": year, "chunk_id": chunk_id,
                            "variable_name": q_code_clean or q_id,
                            "question_id": q_id, "question_code": q_code_clean,
                            "question_text": question_text,
                            "conditional_variable": conditional_var,
                            "instructions_text": instructions_text,
                            "answer_code": answer_code_raw,
                            "answer_text": answer_text_final,
                            "explanation": clean_explanation,
                            "chunk_time_sec": time_sec,
                            "total_chunk_tokens": tokens_used,
                            "value": numeric_value, # Integer representation
                            "order_index": order_index_val
                        })
                    
                    # Advance index by 3 (text, code, val) to next text chunk
                    i += 3
                else:
                    # End of string
                    i += 1

    # -------------------------------------------------------
    # 4. Save CSV Files
    # -------------------------------------------------------
    if all_rows_original:
        df_original = pd.DataFrame(all_rows_original)
        df_original = df_original.sort_values(by=["country", "year", "order_index", "variable_name"])
        if "order_index" in df_original.columns: df_original = df_original.drop(columns=["order_index"])
        df_original.to_csv(output_csv_original, index=False, encoding="utf-8")
        print(f"\n✅ Original CSV created at '{output_csv_original}' ({len(df_original)} rows)")

    if all_rows_dummy:
        df_dummy = pd.DataFrame(all_rows_dummy)
        df_dummy = df_dummy.sort_values(by=["country", "year", "order_index", "variable_name"])
        if "order_index" in df_dummy.columns: df_dummy = df_dummy.drop(columns=["order_index"])
        df_dummy.to_csv(output_csv_dummy, index=False, encoding="utf-8")
        print(f"✅ Dummy CSV created at '{output_csv_dummy}' ({len(df_dummy)} rows)")

        # Pivot
        df_pivot = df_dummy.pivot_table(
            index=["country", "year"], 
            columns="variable_name", 
            values="value", 
            fill_value=0, 
            aggfunc="max"
        ).reset_index()
        
        # Ensure integers in pivot
        int_cols = [c for c in df_pivot.columns if c not in ["country", "year"]]
        df_pivot[int_cols] = df_pivot[int_cols].astype(int)

        df_pivot.to_csv(output_csv_pivot, index=False, encoding="utf-8")
        print(f"✅ Pivoted CSV created at '{output_csv_pivot}' ({len(df_pivot)} rows)")
    else:
        print("\nNo questions found. No CSVs created.")

In [2]:
if __name__ == "__main__":
    # Read API key from file
    try:
        with open("API Key.txt", "r") as f:
            api_key = f.read().strip()
    except FileNotFoundError:
        print("Error: 'API Key.txt' not found. Please create this file with your OpenAI API key.")
        sys.exit(1)
    
    # --- MODIFIED: Define file paths ---
    organized_questions_file = "organized_constitutional_questions.json"
    
    # This path MUST match the output of dependency_scheduler_levels.py
    plan_file = "dependency_plan_levels_organized_constitutional_questions.json"
    # --- END MODIFIED ---
    
    # Check if the plan file exists first
    if not Path(plan_file).exists():
        print(f"Error: Dependency plan '{plan_file}' not found.")
        print(f"Please run `python 3_dependency_scheduler_levels.py` first.")
        sys.exit(1)

    output_directory = "llm_outputs_explanations"

    process_constitutions(
        csv_path="sample_003.csv",
        questions_json_path=organized_questions_file, 
        dependency_plan_path=plan_file, # <--- PASS THE NEW PLAN
        api_key=api_key,
        output_dir=output_directory
    )


Processing United states (1791)...
  - Processing Level 0 (45 questions)...
  - Processing Level 1 (43 questions)...
  - Processing Level 2 (42 questions)...
  - Processing Level 3 (17 questions)...
Processed United states_1791 in 84.03s, tokens=241491

Processing Portugal (1911)...
  - Processing Level 0 (45 questions)...
  - Processing Level 1 (43 questions)...
  - Processing Level 2 (42 questions)...
  - Processing Level 3 (17 questions)...
Processed Portugal_1911 in 75.74s, tokens=312284

Processing Portugal (1933)...
  - Processing Level 0 (45 questions)...
  - Processing Level 1 (43 questions)...
  - Processing Level 2 (42 questions)...
  - Processing Level 3 (17 questions)...
Processed Portugal_1933 in 79.51s, tokens=402297

Processing Portugal (1976)...
  - Processing Level 0 (45 questions)...
  - Processing Level 1 (43 questions)...
  - Processing Level 2 (42 questions)...
  - Processing Level 3 (17 questions)...
Processed Portugal_1976 in 103.01s, tokens=868234

Processing T

In [3]:
if __name__ == "__main__":   
    output_dir = "llm_outputs_explanations" 
    
    convert_json_dir_to_csv(
        json_dir=output_dir, 
        output_csv_original="Port_Taiwan_US_original.csv",
        output_csv_dummy="Port_Taiwan_US_dummy.csv",
        output_csv_pivot="Port_Taiwan_US_pivot.csv"
    )

Found 5 JSON files to process...

✅ Original CSV created at 'Port_Taiwan_US_original.csv' (520 rows)
✅ Dummy CSV created at 'Port_Taiwan_US_dummy.csv' (2729 rows)
✅ Pivoted CSV created at 'Port_Taiwan_US_pivot.csv' (5 rows)


In [36]:
from openai import OpenAI
try:
    with open("API Key.txt", "r") as f:
        api_key = f.read().strip()
except FileNotFoundError:
    print("Error: 'API Key.txt' not found. Please create this file with your OpenAI API key.")
    sys.exit(1)
client = OpenAI(api_key=api_key)

resp = client.responses.create(
    model="gpt-5.1",
    input="Write a short poem about the ocean."
)

print(resp.output_text)


The ocean speaks in languages of blue,  
A restless mirror swallowing the sky.  
Its waves are hands that reach for distant shores,  
Then fall away, unwilling to let go.  

Salt on the wind remembers ancient storms,  
Whales thread their songs through dim cathedral light.  
Beneath the skin of silver, dark and deep,  
The world goes on, unseen but burning bright.  

At dusk, the water gathers up the sun,  
Breaks it in shards along the cooling sand—  
And every night the tides erase our names,  
Then carry them to somewhere we can’t stand.
